# DKT Pipeline — `test.ipynb`

End-to-end Deep Knowledge Tracing pipeline on the student-interaction dataset.

**Task.** For each (user, skill) interaction, predict the evaluation of the *next* attempt as one of `{WRONG, PARTIAL, CORRECT}`.

**Approach.** A per-user sequence model (LSTM) that, at each timestep, emits a 3-way ordinal distribution over the *next* skill the student will attempt. Output is gathered with a one-hot mask of the next skill ID, so we score only the skill the student actually saw next.

**Stages.**
1. Imports & configuration
2. Loading & document universe
3. Cleaning the topic tree
4. Preprocessing transactions
5. Building the feature matrix
6. Splitting and building TensorFlow datasets
7. The DKT model
8. Training
9. Evaluation
10. Closing notes

> Sibling notebook `report.ipynb` runs the same pipeline on a downsampled subset (≤100 users per subject) for fast iteration; this notebook runs on the full data.


## 1. Imports & Configuration

All hyperparameters and dataset-level toggles are concentrated below so the reader can see every knob the pipeline exposes before any code runs. Notably, `MASK_VALUE = -1.0` (not `0.0`) — `prepare_data` pads with `0.0` for the one-hot features and `0` for skill IDs, so a distinct sentinel is reserved if a downstream step needs to mark a *real* masked timestep without colliding with a legitimate zero.


In [1]:
from src.data import *
from src.debug import print_dataset_info
from src.features import *
from src.models import *
from src.models import _flatten

import collections
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn import feature_extraction, model_selection
from sklearn.metrics import mean_squared_error, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import MinMaxScaler


In [2]:
# Pipeline configuration
from src.config import MASK_VALUE

# Topic-tree surgery (see §3 for rationale).
TOPICS_TO_REPARENT = [2026, 2027, 2028]
REPARENT_UNDER     = 1
TOPICS_TO_DROP     = [2029, 3425]

# Modeling toggles.
SUBJECT      = 'math'   # 'math' | 'german'
RANDOM_STATE = 0

# Training / model hyperparameters.
params = {
    'batch_size'        : 32,
    'mask_value'        : MASK_VALUE,
    'recurrent_units'   : 16,
    'dropout_rate'      : 0.1,
    'optimizer'         : 'adam',
    'epochs'            : 20,
    'verbose'           : 1,
    'best_model_weights': 'weights/bestmodel.weights.h5',
}


## 2. Loading & Document Universe

`load_all_data()` returns a `dict` of the five raw tables (`documents`, `topic_trees`, `topics_translated`, `transactions`, `users`). The `documents` table has multiple versions per `document_id`; `get_latest_documents()` keeps the latest version per document and parses the JSON `content` column to extract `estimatedDuration` and `estimatedDifficulty`. `summarize_documents()` reports the share of documents that carry both metadata fields — the rest will be silently dropped at the difficulty join in §4.


In [3]:
dfs = load_all_data()


In [4]:
documents = get_latest_documents(documents=dfs['documents'])
display(documents.head())


,version,document_id,title,type,created_time,content,topic_id,estimatedDuration,estimatedDifficulty
2262,487,fTp7AX3QQfx8vvTLNKGPRv,Aufgabe PT 7.12f,CLOZE_TEXT_DROPDOWN,2018-07-18 14:30:57.539,"{""id"": ""fTp7AX3QQfx8vvTLNKGPRv"", ""type"": ""CLOZ...",1,NaN,NaN
2250,590,fqjEP599AZ29AtN0XNgWxT,Aufgabe PT 5.6,CLUSTER,2018-07-19 13:38:58.442,"{""id"": ""fqjEP599AZ29AtN0XNgWxT"", ""type"": ""CLUS...",1,NaN,NaN
2151,676,etfOjQpk4os8hF9z0rMfwt,Aufgabe 5.10a,MULTIPLE_CHOICE,2018-07-20 10:40:15.845,"{""id"": ""etfOjQpk4os8hF9z0rMfwt"", ""type"": ""MULT...",1,NaN,NaN
1612,677,aVwP8ywckxMbAH-8w2wfMe,Aufgabe PT 5.10d,MULTIPLE_CHOICE,2018-07-20 10:41:00.154,"{""id"": ""aVwP8ywckxMbAH-8w2wfMe"", ""type"": ""MULT...",1,NaN,NaN
1742,680,cEY-K2TJ4HGaNRqp5bjhF5,Aufgabe PT 7.11b,CLOZE_TEXT_DROPDOWN,2018-07-20 11:01:29.324,"{""id"": ""cEY-K2TJ4HGaNRqp5bjhF5"", ""type"": ""CLOZ...",1,NaN,NaN


In [5]:
summarize_documents(documents)


Total distinct documents: 5746
Documents with estimatedDuration and estimatedDifficulty: 4735 (82.4%)
Filtered out: 1011 (17.6%)
Distinct topics: 370


['fTp7AX3QQfx8vvTLNKGPRv',
 'fqjEP599AZ29AtN0XNgWxT',
 'etfOjQpk4os8hF9z0rMfwt',
 'aVwP8ywckxMbAH-8w2wfMe',
 'cEY-K2TJ4HGaNRqp5bjhF5',
 'bKyFkknHAW7aeH4d0qhpEn',
 '1fsDFKDLk8KbZ2xBBzNx--',
 '8JmheE6M4208C3iLPQHRnP',
 '1eUS4GFW4.g8vLxi-sRo1z',
 'bNPQokamA9sbXPqYi8fSzh',
 'awGP3BNZk5UbMHbTW4R5x3',
 '59scgU0wkw-8qdY1ZP59DI',
 'Af0RVdAEibQAr.M9AVTj',
 'ffTij6YYA-zbcx9ZP8QWLy',
 '4HNL62l.47Ta0sHqrOzKoF',
 'aCF0hfb1kQEayDOqwEp0s1',
 '78debrP047o8YoDB0nymp2',
 'dCZ4cWixQrM9ovYwc-TP.z',
 'Bx98geNrvUYWSjsh8g',
 '6f7aujnwQTxb56qwxs1sFX',
 'bVubBjOtA6x9x3pOLGQL95',
 'ctu8cT8ZQVebZn4G3nux8s',
 'dNMQxmWW4pe9auQHzX6DW7',
 '3D4ovRZd4e69keiP-DEyXl',
 '9XwCwbGcQaObYPjBFfQeab',
 'fkcsHMUrkcvadzh3aQdifI',
 '6shnOMCmAMSasjMJay1KdE',
 '448ge.u7Qzyaj4XuEBTIaH',
 '5oWgAX-Fk82bNdaRxoXBgh',
 '6SUe7t6SkV58p70qoJwl3K',
 '9VnHcCno4dF93oGTP1nYyc',
 '483vpC1jQbk8WYeIpt6T-d',
 '56ljt80hkbH8A4vzX.Je5q',
 '3XEDx-kl4Egb92HW9bD0mC',
 '8TxLI5oe4c29OZzvd4o85a',
 'Li7CRHY4gP98s.i4z1-by',
 '5WUsFXhOQwJ9e8Gm81NMto',
 '7CvyCZ

## 3. Cleaning the Topic Tree

The raw topic tree contains structural noise that hurts the modeling task:
- Several top-level topics that should logically sit under the German root.
- Branches that no document references (dead leaves and dead intermediate nodes).
- A pair of subtrees that are explicitly stub/placeholder content.

We perform three surgical operations in order, each justified below.


### 3.1 Inspect the raw tree

`build_topic_lookups()` returns four dicts (`id_to_name`, `id_to_math`, `child_to_parent`, `parent_to_children`). `add_topic_depth()` copies `topics_translated` and adds a `depth` column by walking parents. We use the depth distribution and per-topic document counts to identify candidates for surgery.


In [6]:
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics  = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])
print(f"\n{len(lookups['child_to_parent'])} child→parent edges")


Depth distribution across all topics:
       n_topics
depth          
0            25
1            72
2           179
3           294
4           110
5            20

675 child→parent edges


In [7]:
display(docs_per_topic(documents).head(20))
display(docs_per_depth(documents, topics))
display(topics_with_docs_per_depth(documents, topics))


,n_documents
topic_id,
1,319
998,71
988,67
1040,65
955,64
2069,63
1032,62
975,56
956,55


,n_documents
depth,
0,388
1,126
2,1606
3,3348
4,145
5,133


,n_topics_with_docs
depth,
0,7
1,20
2,98
3,187
4,38
5,20


In [8]:
# Depth-0 topics that actually carry documents — candidates for the root or for surgery.
active = documents['topic_id'].unique()
display(topics[(topics['depth'] == 0) & (topics['id'].isin(active))])


,id,german_name,german_description,name,description,math,depth
0,1,Deutsch,Sprache als System,German,Language as a system,0,0
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1,0
116,2026,Ober-/Unterbegriff,NaN,Top/sub term,NaN,0,0
117,2027,Synonyme/Antonyme,NaN,Synonyms/antonyms,NaN,0,0
118,2028,Mehrdeutigkeit,NaN,Ambiguity,NaN,0,0
119,2029,Verschiedenes,NaN,Miscellaneous,NaN,0,0
477,3425,Zu löschen,"ungültig, wird gelöscht.",To delete,"invalid, will be deleted.",1,0


### 3.2 Reparent `[2026, 2027, 2028]` under root `1`

<!-- TODO(christophe): rationale for reparenting these three German topics under the German root -->

`reparent_topics()` ([src/data.py:146](src/data.py#L146)) detaches the listed children and re-inserts them with a new parent, allocating fresh `topic_id`s past the current max so primary keys stay unique.


In [9]:
dfs['topic_trees'] = reparent_topics(
    dfs['topic_trees'], TOPICS_TO_REPARENT, new_parent_id=REPARENT_UNDER,
)
display(dfs['topic_trees'][dfs['topic_trees']['child_id'].isin(TOPICS_TO_REPARENT)])


,topic_id,parent_id,child_id,sibling_rank,displayed_on_dashboard
678,6108,1.0,2026,0,0
679,6109,1.0,2027,0,0
680,6110,1.0,2028,0,0


### 3.3 Prune topics with no documents

`_non_empty_topic_ids` ([src/data.py:167](src/data.py#L167)) starts from the set of topics that *directly* host at least one document and walks parent pointers to collect every ancestor. A topic survives iff it has a descendant document; this drops dead leaves *and* dead intermediate nodes in one pass. We apply the same filter to both `topic_trees` and `topics_translated` so the two stay in sync.


In [10]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees']        = prune_empty_topics(dfs['topic_trees'], documents)
dfs['topics_translated']  = prune_empty_topics_translated(
    dfs['topics_translated'], dfs['topic_trees'], documents,
)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")


topic_trees rows: 681 -> 378
topics_translated rows: 700 -> 379


### 3.4 Drop the `[2029, 3425]` subtrees

<!-- TODO(christophe): rationale — likely "Miscellaneous" + "To delete" stub branches that survived the empty-prune because they still host one or two stale documents -->

`drop_topic_subtrees()` ([src/data.py:213](src/data.py#L213)) does a downward BFS from each root and removes every collected ID from both tables.


In [11]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees'], dfs['topics_translated'] = drop_topic_subtrees(
    dfs['topic_trees'], dfs['topics_translated'], TOPICS_TO_DROP,
)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")


topic_trees rows: 378 -> 377
topics_translated rows: 379 -> 377


In [12]:
# Canonical lookups + depth table after all surgery — used by everything downstream.
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics  = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])


Depth distribution across all topics:
       n_topics
depth          
0             2
1            25
2           103
3           188
4            39
5            20


## 4. Preprocessing Transactions

`summarize_transactions()` ([src/data.py:240](src/data.py#L240)) does two things at once:
1. Keeps only rows with a non-null `evaluation` (drops in-progress / abandoned attempts).
2. Joins each transaction's `estimatedDifficulty` from `documents` via `document_id`.

It also reports two leakage sources — rows whose `topic_id` is no longer in the cleaned topics table, and rows whose document has no `estimatedDifficulty`. **No rows are dropped here**: filtering is deferred to feature construction, where missing values are coalesced to a `-2` sentinel that becomes part of the skill key.


In [13]:
evaluated = summarize_transactions(dfs['transactions'], topics, documents)
display(evaluated.head())


Total transactions: 2134759
Evaluated transactions: 1401007 (65.6%)
Evaluated with unknown topic_id: 348782 (24.9%)
Evaluated with no estimatedDifficulty: 34228 (2.4%)


,transaction_id,transaction_token,user_id,document_id,document_version,evaluation,input,start_time,commit_time,user_agent,...,session_id,topic_id,session_closed,session_type,session_accepted,challenge,challenge_id,challenge_order,challenge_name,estimatedDifficulty
0,688413,88fdcaad-f73b-46a2-b561-d262f2441442,393211,awd0i1DlVtg6kuMZSkpmHa,75002,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 07:58:27.312000000,2021-05-21 08:03:43.020000000,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,2.0,G3h – Training Rhetorik,3.0
1,688414,a75eb7b4-b2c2-47d4-9200-27980c175037,393211,arhWF3BT53V9W8cGOaZVPX,75012,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 08:04:05.067000000,2021-05-21 08:07:21.288999936,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,3.0,G3h – Training Rhetorik,4.0
2,688415,61eb829d-bdda-4107-86af-ad9a14a7bdc9,393211,9wk5dtV2mF59odW0wCEYYc,75003,PARTIAL,"{""type"": ""CLOZE_TEXT"", ""clozeInputs"": [""Person...",2021-05-21 08:07:37.048000000,2021-05-21 08:13:30.953999872,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,4.0,G3h – Training Rhetorik,3.0
3,688416,30ff0d8a-865d-460b-9177-b698a52b0d5c,393211,afilxZ8LycP5LReULeKngW,75009,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Ich gehe i...",2021-05-21 08:13:38.943000000,2021-05-21 08:22:13.975000064,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,5.0,G3h – Training Rhetorik,3.0
4,688417,0adedf3b-ba35-4497-8c6b-b5c2f6fcbbf3,393211,76m6v05NCeX8x2Wr5tKRE3,75007,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Kleiner Ma...",2021-05-21 08:22:19.391000000,2021-05-21 08:22:55.366000128,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,6.0,G3h – Training Rhetorik,2.0


### 4.1 Split by subject

`split_by_subject()` partitions on `topics.math ∈ {0, 1}`. We pick one subject for the rest of the pipeline; flipping `SUBJECT` in §1 reruns everything against German.


In [14]:
math_df, german_df = split_by_subject(evaluated, topics)
print(f"math: {len(math_df)} rows | german: {len(german_df)} rows")
df = math_df if SUBJECT == 'math' else german_df


math: 293465 rows | german: 758760 rows


## 5. Building the Feature Matrix

`build_feature_matrix()` ([src/features.py:10](src/features.py#L10)) produces the modeling table `X` with columns `[user_id, skill_name, correct, start_time, skill_attempts, total_attempts]`.

Three non-obvious choices to call out:
- **Composite skill key.** `skill_name = topic_id + '_' + estimatedDifficulty` — same topic at a different difficulty is treated as a *different* skill. This is what lets the DKT model learn distinct mastery curves per difficulty band.
- **Sentinel `-2` for missing values.** Both `topic_id` and `estimatedDifficulty` get `fillna(-2)` *before* the skill key is built, so missing-metadata rows form their own pseudo-skills (e.g. `-2_3`, `123_-2`) instead of being dropped. The downstream model sees them as just another skill.
- **Causal cumulative counters.** `skill_attempts` and `total_attempts` use `cumcount()` per group, so the count at row *i* reflects *only* attempts strictly before *i* — no leakage of the current row.

The label column `correct` is mapped from `evaluation` strings via `EVAL_CODES = {WRONG: 0, PARTIAL: 1, CORRECT: 2}` ([src/features.py:1](src/features.py#L1)) inside the function — the call site stays clean.


In [15]:
X = build_feature_matrix(df, documents)
display(X.head())


,user_id,skill_name,correct,start_time,skill_attempts,total_attempts
0,390142,1046_2,2,2021-05-21 10:16:09.924000000,0,0
1,390137,1059_1,1,2021-05-21 10:16:58.803000000,0,0
2,390140,987_1,0,2021-05-21 10:20:53.223000000,0,0
3,390140,987_2,0,2021-05-21 10:23:24.005000000,0,1
4,390140,987_1,0,2021-05-21 10:24:10.729000000,1,2


In [16]:
print("Number of unique students in the dataset:", X['user_id'].nunique())
print("Number of unique skills in the dataset:",  X['skill_name'].nunique())


Number of unique students in the dataset: 9110
Number of unique skills in the dataset: 157


## 6. Splitting and Building TensorFlow Datasets

Three subtleties drive this section:

**Group-aware split.** We split by *user*, not by interaction, using `GroupShuffleSplit` ([src/models.py:10](src/models.py#L10)) with `random_state=RANDOM_STATE`. If we naively `train_test_split`-ed rows, the same student would appear in both folds and the model would memorize per-user difficulty rather than learn skill mastery dynamics. We apply the same split twice (80/20 outer, then 80/20 inside train) to get an effective **64/16/20 train/val/test** breakdown.

**Sequence construction.** `prepare_seq()` ([src/features.py:45](src/features.py#L45)) factorizes `skill_name` with `sort=True` (alphabetical, deterministic), then for each user produces the DKT triple:
- `past skill_with_answer` = `(skill_id × 3 + correct)` shifted to `[:-1]` — what the student saw at each step, encoded jointly.
- `next skill` = `skill_id` shifted to `[1:]` — the skill we want to predict next.
- `next correct` = `correct` shifted to `[1:]` — the label.

**Padding & repetition.** `prepare_data()` ([src/models.py:23](src/models.py#L23)) one-hot encodes the past-features tensor, pads each batch to its longest sequence (`drop_remainder=True` to keep static shapes), and `repeat()`s the dataset. An epoch is bounded by `steps_per_epoch`, computed as `len(seq) // batch_size`.


In [17]:
# 80/20 outer split, then 80/20 inside train. create_iterator already lives in src/models.py.
train_index, test_index = next(create_iterator(X))
X_train, X_test         = X.iloc[train_index], X.iloc[test_index]

train_val_index, val_index = next(create_iterator(X_train))
X_train_val, X_val         = X_train.iloc[train_val_index], X_train.iloc[val_index]


In [18]:
# Build TF datasets for train / val / test.
seq, features_depth, skill_depth = prepare_seq(X)
seq_train = seq[X_train_val.user_id.unique()]
seq_val   = seq[X_val.user_id.unique()]
seq_test  = seq[X_test.user_id.unique()]

tf_train, length      = prepare_data(seq_train, params, features_depth, skill_depth)
tf_val,   val_length  = prepare_data(seq_val,   params, features_depth, skill_depth)
tf_test,  test_length = prepare_data(seq_test,  params, features_depth, skill_depth)

params['train_size'] = int(length      // params['batch_size'])
params['val_size']   = int(val_length  // params['batch_size'])
params['test_size']  = int(test_length // params['batch_size'])

print({k: params[k] for k in ('batch_size', 'train_size', 'val_size', 'test_size')})


{'batch_size': 32, 'train_size': 182, 'val_size': 45, 'test_size': 56}


2026-04-27 23:29:56.271298: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-04-27 23:29:56.271958: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-27 23:29:56.272825: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-27 23:29:56.273233: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-27 23:29:56.274151: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 7. The DKT Model

`create_model_lstm()` ([src/models.py:93](src/models.py#L93)) wires up:

```
features (B, T, F) ─► LSTM(recurrent_units, return_sequences=True, dropout)
                  ─► TimeDistributed(Dense(n_skills × N_EVAL_STATES))
                  ─► Reshape to (B, T, n_skills, N_EVAL_STATES)
                  ─► GatherSkill: einsum with one-hot(next_skill)
                  ─► output (B, T, N_EVAL_STATES)
```

The `GatherSkill` layer ([src/models.py:79](src/models.py#L79)) is the trick that makes the loss focused: at each timestep we emit a 3-vector for *every* skill, but the loss is computed only on the row corresponding to the skill the student actually attempted next.

**Metrics.**
- `AUC`: macro one-vs-rest across the 3 ordinal classes (multi-label mode on the softmaxed logits).
- `RMSE`: ordinal RMSE between the true class index `{0, 1, 2}` and the softmax-expected ordinal value. This rewards being *close* (predicting `PARTIAL` for `CORRECT` is better than predicting `WRONG`).
- `accuracy`: plain sparse categorical accuracy on the argmax.


In [19]:
dkt_lstm = create_model_lstm(features_depth, skill_depth, params)
dkt_lstm.summary()


Model: "DKT"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 features (InputLayer)       [(None, None, 471)]          0         []                            
                                                                                                  
 lstm (LSTM)                 (None, None, 16)             31232     ['features[0][0]']            
                                                                                                  
 time_distributed (TimeDist  (None, None, 468)            7956      ['lstm[0][0]']                
 ributed)                                                                                         
                                                                                                  
 reshape (Reshape)           (None, None, 156, 3)         0         ['time_distributed[0][0]']  

In [20]:
# Quick shape check before kicking off a 20-epoch training run.
(inputs, label, weight) = next(iter(tf_train))
y = dkt_lstm(inputs, training=False)
assert y.shape[-1] == N_EVAL_STATES, f"unexpected head dim: {y.shape}"
print("forward pass OK:", y.shape)


forward pass OK: (32, 186, 3)


## 8. Training

`train_dkt()` ([src/models.py:53](src/models.py#L53)) wires up a `ModelCheckpoint` that saves only the best-validation weights, and runs `model.fit` against the repeating `tf_train` / `tf_val` streams. Two details:
- We use `steps_per_epoch = train_size` directly (one full pass through the unique batches per epoch).
- 20 epochs is empirically enough for AUC to plateau around ~0.71 on the math subject.


In [ ]:
history = train_dkt(dkt_lstm, tf_train, tf_val, params)


Epoch 1/20


2026-04-27 23:29:58.039085: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


182/182 [==============================] - 54s 287ms/step - loss: 0.1745 - auc: 0.6042 - root_mean_squared_error: 0.8726 - accuracy: 0.5563 - val_loss: 0.1595 - val_auc: 0.6539 - val_root_mean_squared_error: 0.8426 - val_accuracy: 0.5641
Epoch 2/20
171/182 [===========================>..] - ETA: 3s - loss: 0.1582 - auc: 0.6651 - root_mean_squared_error: 0.8410 - accuracy: 0.5732

## 9. Evaluation

We reload the best-validation weights and score on the held-out test users. `model.evaluate` returns the headline metrics; below it we build a per-class confusion matrix to inspect the *shape* of the model's errors — for an ordinal task, off-by-one errors (e.g. predicting `PARTIAL` for `CORRECT`) are far less concerning than flipping `WRONG ↔ CORRECT`.


In [ ]:
dkt_lstm.load_weights(params['best_model_weights'])
dkt_lstm.evaluate(tf_test, steps=params['test_size'], verbose=params['verbose'], return_dict=True)


### 9.1 Confusion matrix (row-normalized)

Rows are true classes; each row sums to 1. The diagonal is per-class recall; off-diagonal cells reveal which substitutions the model is making.


In [ ]:
true_all, pred_all = [], []
for (inputs, lbl, w) in tf_test.take(params['test_size']):
    keep = w.numpy() > 0
    pred = tf.argmax(dkt_lstm(inputs), axis=-1).numpy()
    true_all.extend(lbl.numpy()[keep].tolist())
    pred_all.extend(pred[keep].tolist())

true_all = np.array(true_all); pred_all = np.array(pred_all)
names = ['WRONG', 'PARTIAL', 'CORRECT']

print("true distribution:", collections.Counter(true_all))
print("pred distribution:", collections.Counter(pred_all))

cm = pd.crosstab(
    pd.Series(true_all, name='true'),
    pd.Series(pred_all, name='pred'),
    normalize='index',
).round(3)
cm.index   = [names[i] for i in cm.index]
cm.columns = [names[i] for i in cm.columns]
print(cm)


## 10. Closing Notes

- **Reproducibility.** Splits are deterministic via `RANDOM_STATE`, but training on Apple's Metal GPU is not bit-exact across runs — expect ±0.01 AUC noise.
- **Subject toggle.** Switching `SUBJECT = 'german'` in §1 reruns the entire pipeline against German interactions; no other code change is needed.
- **Future work.** Richer features (response time, time-of-day, attempt streak length), per-user calibration of the ordinal head, and skill-frequency reweighting in the loss are the natural next steps.
